# Topic 05 — Solutions: GroupBy & Aggregation

*Reference solutions for the objective tasks. Try the practice first.* The Warm-Up concept questions, Interview Lens and Reflection have **no key** — by design.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
oi = items.merge(orders[['order_id','channel','customer_id']], on='order_id', how='left')


### Warm-Up — NumPy recall (self-check)

In [ ]:
keys = np.array(['a','b','a','b','a']); vals = np.array([1.,2.,3.,4.,5.])
sum_a = vals[keys=='a'].sum()
print(sum_a)

### Core lesson tasks

In [ ]:
channel_rev = oi.groupby('channel')['line_revenue'].sum()
print(channel_rev.sort_values(ascending=False))

In [ ]:
summary = items.groupby('product_id').agg(revenue=('line_revenue','sum'), units=('quantity','sum'), lines=('order_id','count'))
print(summary.sort_values('revenue', ascending=False).head())

In [ ]:
items['pct_of_product'] = items['line_revenue'] / items.groupby('product_id')['line_revenue'].transform('sum')
print(items.groupby('product_id')['pct_of_product'].sum().dropna().mean())

### Mixed review (earlier topics)

In [ ]:
print(orders['channel'].nunique(), orders['channel'].unique())

In [ ]:
oi2 = oi.merge(orders[['order_id','status']], on='order_id', how='left')
delivered_rev = oi2.loc[oi2['status']=='delivered','line_revenue'].sum()
print(f'£{delivered_rev:,.0f}')

### Data detective

In [ ]:
order_rev = oi.groupby('order_id')['line_revenue'].sum()
print('AOV', round(order_rev.mean(),2), '| top', oi.groupby('customer_id')['line_revenue'].sum().idxmax())

### Interview Lens — discussion points (not full answers)
- **Q14:** split rows by key → reduce each group → combine to a table.
- **Q16:** agg=one value/group; transform=same-length broadcast; apply=arbitrary per-group function (slowest).